# Clustering: K-means & Hierarchical Agglomerative Clustering

This project applies two unsupervised clustering algorithms — K-means and Hierarchical Agglomerative Clustering (HAC) — to the Abalone, Iris, and white wine quality datasets. It covers initial-centroid sensitivity in K-means, silhouette-based evaluation across cluster counts, comparison of HAC linkage methods, and applying both algorithms to an unfamiliar dataset.

In [ ]:
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score
import numpy as np
import pandas as pd
from scipy.io import arff

## K-means and HAC on the Abalone Dataset

K-means (`n_clusters=3, init='random', n_init=1`) and HAC (complete linkage, k=3) applied to the Abalone dataset, treating the output class as an additional input feature.

In [ ]:
# K-means with Abalone
train_data,_ = arff.loadarff("data/abalone.arff")

abalone_df = pd.DataFrame(train_data)

# Quick EDA
print("First 5 rows of Abalone data set:")
display(abalone_df.head())

print("\nAbalone Feature Distribution:")
display(abalone_df.value_counts().sort_index()) # here we see lots of unique data

print(f"Data shape: {abalone_df.shape}")

# no train test split since this is all exploratory
# create the model

clusters = 3

kmeans_abalone = KMeans(n_clusters = clusters, init = "random", n_init = 1)
kmeans_abalone.fit(abalone_df)

labels = kmeans_abalone.labels_
centers = kmeans_abalone.cluster_centers_
n_iterations = kmeans_abalone.n_iter_
total_sse = kmeans_abalone.inertia_

# reshape data so silhouette score can be calculated
reshaped_train_data = train_data.view(np.float64).reshape(train_data.shape[0], -1)
avg_silhouette = silhouette_score(reshaped_train_data, labels)

# format and display all metrics
metrics = {
    "Metric": ["Number of Iterations", "Total SSE (Inertia)", "Avg Silhouette Score"],
    "Value":  [n_iterations, round(total_sse, 4), round(avg_silhouette, 4)]
}
print(f"\nKMeans Model Metrics (Abalone) with n_clusters = {clusters}:")
display(pd.DataFrame(metrics))

# cluster centers in their own table
centers_df = pd.DataFrame(
    centers,
    columns=abalone_df.columns,
    index=[f"Cluster {i}" for i in range(len(centers))]
)
print("\nCluster Centers:")
display(centers_df)

# labels as a compact series
print("\nPoint Labels:")
labels_df = pd.DataFrame({"Point": range(len(labels)), "Cluster Label": labels})
display(labels_df)

**Results**

For `n_clusters = 3`, the initial run had a silhouette score of 0.5184. Re-running with different values of *k* gave:

| k | Silhouette Score |
|---|---|
| 3 | 0.5184 |
| 5 | 0.5095 |
| 7 | 0.4751 |
| 9 | 0.5091 |
| 11 | 0.5025 |

A silhouette score measures how well each point fits its assigned cluster, ranging from -1 (poor fit) to 1 (strong fit). The consistency around 0.5 across every value of *k* suggests there isn't a clear "best" number of clusters for this data — likely because K-means assumes roughly spherical clusters, and Abalone's features (age and size) form more of a continuous gradient than distinct groups. As *k* increased, the model simply subdivided that gradient into more pieces rather than finding meaningfully different structure.

In [ ]:
# HAC with Abalone
# See cells above for a EDA on this data set.

# Reload data because I think I messed with it too much in the section above
train_data,_ = arff.loadarff("data/abalone.arff")
abalone_df = pd.DataFrame(train_data)

# initialize and fit the model
hac = AgglomerativeClustering(n_clusters = 3, linkage = "complete")
hac.fit(abalone_df)

# get the metrics needed
labels = hac.labels_
avg_silhouette = silhouette_score(abalone_df, labels)

metrics = {
    "Metric": "Avg Silhouette Score",
    "Value":  round(avg_silhouette, 4)
}
print(f"\nAgglomerative Clustering (Abalone) Silhouette Score:")
display(pd.DataFrame([metrics]))

# labels as a compact series
print("\nPoint Labels for Agglomerative Clustering:")
labels_df = pd.DataFrame({"Point": range(len(labels)), "Cluster Label": labels})
display(labels_df)

# manually compute the centers so that we can compare with them
hac_centers = np.array([
    abalone_df[labels == i].mean(axis=0) 
    for i in range(3)
])

centers_df = pd.DataFrame(
    hac_centers,
    columns=abalone_df.columns,
    index=[f"Cluster {i}" for i in range(3)]
)
display(centers_df)


**Results**

HAC's silhouette score came out slightly higher than any K-means run above. The two algorithms also produce meaningfully different cluster centers: K-means assumes roughly circular clusters and starts from randomly chosen centers, correcting iteratively — which introduces some run-to-run variation and works best when clusters are genuinely spherical. HAC is deterministic and builds clusters bottom-up from pairwise distances, making no assumption about cluster shape.

On this dataset, HAC's higher silhouette score suggests it does a better job matching points to sensible groups, likely because it isn't constrained by K-means' spherical-cluster assumption.

## K-means Initial Centroid Experiments

K-means results can vary depending on the initial centroids. This section compares five manual runs with random initial centroids (`n_init=1`), against `n_init=5` (best of 5 automatic runs), against K-means++ initialization.

In [ ]:
# K-means initial centroid experiments

# Load the iris data set
train_data, _ = arff.loadarff("data/iris.arff")
iris_df = pd.DataFrame(train_data)

# drop the class from the training data
X = iris_df.drop(["class"], axis=1)

# 1. 
# manual runs 

# lists for my data frame later
inertias = []
sil_scores = []

# initialize my models and get their metrics
for _ in range(1, 6):
    kmeans = KMeans(n_clusters=4, init="random", n_init=1)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X, kmeans.labels_))

runs = {
    "Run": range(1, 6),
    "Inertia": inertias,
    "Silhouette Score": sil_scores
}
runs = pd.DataFrame(runs)
print("\nManual Runs (random init, n_init=1)")
display(runs)

print("\nAverage Inertia and Silhouette Scores for manual runs:")
averages = {
    "Average Inertia": sum(inertias)/len(inertias),
    "Average Silhouette Score": sum(inertias)/len(inertias)
}
averages = pd.DataFrame([averages])
display(averages)

# 2. 
# n_init
kmeans_ninit = KMeans(n_clusters=4, init="random", n_init=5)
kmeans_ninit.fit(X)

# data tableee
sil_score = silhouette_score(X, kmeans_ninit.labels_)
ninit_metrics = {
    "Inertia": [kmeans_ninit.inertia_],
    "Silhouette Score": [sil_score]
}
ninit_metrics = pd.DataFrame(ninit_metrics)
print("\nn_init=5 (best of 5 automatic runs)")
display(ninit_metrics)

# 3. 
# KMeans++ with n_init=1
kmeans_pp = KMeans(n_clusters=4, init="k-means++", n_init=1)
kmeans_pp.fit(X)

sil_score = silhouette_score(X, kmeans_pp.labels_)
pp_metrics = {
    "Inertia": [kmeans_pp.inertia_],
    "Silhouette Score": [sil_score]
}
pp_metrics = pd.DataFrame(pp_metrics)
print("\nK-Means++ (n_init=1)")
display(pp_metrics)

**Results**

A silhouette score reflects how well points fit their assigned clusters (closer to 1 is better); inertia reflects cluster compactness (lower is better).

Of the five manual runs, the first had noticeably worse inertia and silhouette scores than the other four, which converged to nearly identical results — suggesting there's a clear "best" solution for this data that multiple random initializations tend to find. Both `n_init=5` and K-means++ achieved results nearly identical to runs 2–5 of the manual trials, just via different strategies.

A single random run (`n_init=1`) is unreliable, since an unlucky initialization can converge to a worse local optimum. Increasing `n_init` addresses this by brute force — running multiple times and keeping the best result. K-means++ is a more efficient solution: it runs a quick preliminary pass to choose good initial centroids, then runs K-means once from that better starting point.

## Silhouette Graphs Across k (Iris)

Silhouette graphs for K-means with k = 2–6 on the Iris dataset, using the default `n_init=10`.

In [ ]:
# Iris Clustering with K-means and silhouette graphs
import sys
import setuptools._distutils as distutils
sys.modules["distutils"] = distutils
# I had to add these random import statements above because I couldn't get yellowbrick to work
# even after downgrading the version of python that I am using
from yellowbrick.cluster import SilhouetteVisualizer
import matplotlib.pyplot as plt

# to see an EDA, see previous cells
for k in range(2, 7):
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # initialize KMeans model
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    
    # Create and fit the SilhouetteVisualizer
    visualizer = SilhouetteVisualizer(kmeans, colors='yellowbrick', ax=ax, force_model = True)
    visualizer.fit(X)
    
    # Print SSE (inertia) and silhouette score
    print(f"k={k}")
    print(f"SSE (Inertia): {kmeans.inertia_:.2f}")
    print(f"Silhouette Score: {visualizer.silhouette_score_:.4f}")
    
    visualizer.finalize()
    plt.tight_layout()
    plt.show()
    plt.clf()

**Results**

A good clustering, based on these plots, shows: a high average silhouette score, no negative values, roughly balanced cluster sizes, a gradual taper rather than sharp drop-offs, and a meaningful inertia score.

| k | Inertia | Silhouette Score |
|---|---|---|
| 2 | 152.37 | 0.6808 |
| 3 | 78.94 | 0.5526 |
| 4 | 57.32 | 0.4978 |
| 5 | 46.54 | 0.4885 |
| 6 | 38.93 | 0.3682 |

**k = 2** is the clear standout: a high average score, two distinct and roughly balanced clusters, no negative values, and a gradual descent. From k = 3 onward, every metric degrades — negative values appear at k = 4–6 (indicating misclustered points), cluster sizes become unbalanced (especially at k = 5), and the smooth gradient seen at k = 2 disappears in favor of jagged, irregular plots. **k = 2 is the best choice** for this dataset by every criterion examined.

## HAC Across Linkage Types (Iris)

Silhouette scores for HAC on the Iris dataset across k = 2–6, comparing single, average, complete, and ward linkage.

In [ ]:
#HAC with Iris
# again, for eda information, please look at the cells above :) I don't want to be redundant.

# all of the linkage things to be used here in a second
linkages = ["single", "average", "complete", "ward"]

# results list
results = []

# for loop to get everything set up
for linkage in linkages:
    for k in range(2, 7):
        # initialize the model
        hac = AgglomerativeClustering(n_clusters = k, linkage = linkage)
        hac.fit(X)

        # labels, sihlouette scores, and appending to results
        labels = hac.labels_
        sil_score = silhouette_score(X, labels)
        results.append({"Linkage": linkage, "n_clusters": k, "Silhouette Score":round(sil_score, 4)})

results = pd.DataFrame(results)

print("Silhouette Scores for a given model with varying k and linkage:")
display(results)

**Results**

HAC's results track closely with the K-means results above (k = 2 SSE: 152.37, Silhouette: 0.6808; k = 3: 78.94 / 0.5526; k = 4: 57.32 / 0.4978; k = 5: 46.54 / 0.4885; k = 6: 38.93 / 0.3682) — k = 2 performs best regardless of algorithm or linkage method, and scores decline as k increases across the board. That consistency across two very different algorithms suggests there's genuine underlying structure in the data that both are picking up on, rather than an artifact of either method.

Among linkage methods: complete linkage performs notably worse at k = 2. Single linkage consistently underperforms across all k values. Ward and average linkage are the most consistent and strongest performers, degrading gradually rather than dropping sharply. Overall, values of k above 3 aren't worth using here — scores fall off meaningfully beyond that point.

## Applying Both Algorithms to a New Dataset

K-means and HAC applied to the white wine quality dataset (features scaled via `StandardScaler`), comparing several parameter configurations for each algorithm.

In [ ]:
from sklearn.preprocessing import StandardScaler
# Run both algoriths on a data set of your choice

# Load the data
wine_df = pd.read_csv("data/winequality-white.csv", sep=';')

# Quick EDA
print("Quick EDA:")
print("\nFirst 5 rows of the Wine Quality dataset:")
display(wine_df.head())

print("\nWine Quality Distribution:")
display(wine_df["quality"].value_counts().sort_index())

# X, decided to drop the quality part
X = wine_df.drop(["quality"], axis = 1)

# scale the dataset so clustering can work the way it is supposed to
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# do kmeans first
print("\nKMeans Results:")

km1 = KMeans(n_clusters=2, n_init=10, random_state=42)
km2 = KMeans(n_clusters=3, n_init=10, random_state=42)
km3 = KMeans(n_clusters=4, n_init=10, random_state=42)

km_results = []

for km in [km1, km2, km3]:
    # fit
    km.fit(X_scaled)

    # get metrics
    labels = km.labels_
    sil_score  = silhouette_score(X_scaled, labels)
    inertia = km.inertia_

    # add to results list
    km_results.append({"n_clusters:" : km.n_clusters, "Inertia:": inertia, "Silhouette Scores": sil_score})

km_results = pd.DataFrame(km_results)
display(km_results)

# then hac
print("\nHAC Results:")

hac1 = AgglomerativeClustering(n_clusters=2, linkage="ward")
hac2 = AgglomerativeClustering(n_clusters=3, linkage="ward")
hac3 = AgglomerativeClustering(n_clusters=3, linkage="average")

hac_results = []

for hac in [hac1, hac2, hac3]:
    # fit
    hac.fit(X_scaled)

    # get metrics
    labels = hac.labels_
    sil_score = silhouette_score(X_scaled, labels)
    linkage = hac.linkage

    # add to results list
    hac_results.append({"n_clusters:" : hac.n_clusters, "Linkage:": linkage, "Silhouette Scores": sil_score})

hac_results = pd.DataFrame(hac_results)
display(hac_results)

# the graph!! 
fig, ax = plt.subplots(figsize=(10, 6))
visualizer = SilhouetteVisualizer(km1, colors='yellowbrick', ax=ax, force_model=True)
visualizer.fit(X_scaled)
ax.set_title("Silhouette Plot - k=3")
visualizer.finalize()
plt.show()




**Results**

K-means followed the same pattern seen with Iris — silhouette scores decreased as k increased — but started from a much weaker baseline: k = 2 scored only around 0.2, a poor result. That, combined with the uneven cluster sizes and negative values visible in the silhouette plot, suggests K-means is not well suited to this dataset (or that better hyperparameters exist than the ones tried here).

HAC told a more interesting story. Ward linkage at both k = 2 and k = 3 scored similarly poorly to K-means — expected, since ward linkage optimizes a similar within-cluster-distance objective. But average linkage at k = 3 reached a silhouette score of 0.56, substantially higher than any other configuration tested. Average linkage is more flexible about cluster shape than ward or K-means' spherical assumption, which likely explains why it found a meaningfully better grouping in data that probably doesn't form clean spherical clusters.

**HAC with average linkage and k = 3** was the best-performing configuration for this dataset.

## Conclusion

Across all three datasets, K-means and HAC produced remarkably consistent silhouette trends — a strong signal that the underlying cluster structure (or lack thereof) is a property of the data rather than an artifact of any one algorithm. The clearest result came from Iris, where k = 2 was unambiguously best across every metric. The most interesting result came from the wine quality dataset, where switching from ward to average linkage in HAC produced a substantially better clustering — a reminder that linkage choice matters as much as the algorithm itself when clusters aren't spherical.

A natural next step would be revisiting the Abalone and wine datasets with density-based methods like DBSCAN, which make no assumptions about cluster shape at all.